# Study 936 — Tolerance Bands — the teardown

One book, four schedules, one execution lag. The excess-of-cash race, the Newey-West *t* on the daily return difference, paired block-bootstrap CIs on the Sharpe difference, the era cut, the cost and band-width sweeps, the 23-year cross-check, and the live synthetic control.

**Design.** All four schedules hold the *identical* sleeves in the *identical* target mix, so the market factor and the cash leg cancel exactly in every pairwise **return** difference. That difference series is therefore the clean statistic: it isolates *when each rule trades* from *what the book owns*. (Careful: a Sharpe **ratio** is nonlinear, so the *Sharpe* difference is **not** cash-invariant — excess-of-cash and gross-of-cash Sharpe gaps differ by ~0.008 here. Sharpe gaps are only ever compared on a matched cash basis.) Signal read at the close of day *t*, executed at the close of *t+1* — one lag, nowhere else. Cost is one-way bps x traded notional / NAV; nothing is shorted, so no borrow leg. **Tax is not modelled** and is the one omitted cost that is first-order relative to the effects measured here.

Every real number is frozen from `docs/results.md` (returns fingerprint `958f652ce4c9`; the levels fingerprint `aa48518ec578` is printed too but churns on every dividend re-adjustment, so it is a diagnostic, not the key), as-of 2026-06-30.

In [1]:
R = {'start': '2007-05-30', 'end': '2026-06-30', 'n_days': 4802, 'fp': '958f652ce4c9', 'fp_levels': 'aa48518ec578', 'a_drift_s': 0.6788, 'a_ann_s': 0.6673, 'a_qtr_s': 0.657, 'a_bnd_s': 0.6412, 'a_drift_c': 7.23, 'a_ann_c': 6.79, 'a_qtr_c': 6.84, 'a_bnd_c': 6.88, 'a_drift_v': 11.22, 'a_ann_v': 10.71, 'a_qtr_v': 11.0, 'a_bnd_v': 11.4, 'a_drift_dd': -24.92, 'a_ann_dd': -30.01, 'a_qtr_dd': -31.74, 'a_bnd_dd': -31.86, 'a_ann_to': 0.079, 'a_qtr_to': 0.157, 'a_bnd_to': 0.114, 'a_ann_n': 19, 'a_qtr_n': 76, 'a_bnd_n': 22, 'a_diff_ba': -0.0261, 'a_t_ba': 0.82, 'a_cagr_ba': 0.092, 'a_diff_bq': -0.0158, 'a_t_bq': 0.64, 'a_diff_qa': -0.0103, 'a_t_qa': 0.53, 'a_diff_ba_gross': -0.034, 'a_diff_bq_gross': -0.0204, 'a_diff_qa_gross': -0.0136, 'a_ci_lo': -0.0674, 'a_ci_hi': 0.0142, 'a_ci_neg': 87.9, 'b_drift_s': 0.7393, 'b_ann_s': 0.7718, 'b_qtr_s': 0.7559, 'b_bnd_s': 0.7441, 'b_diff_ba': -0.0277, 'b_t_ba': 0.8, 'b_cagr_ba': 0.122, 'b_ci_lo': -0.0685, 'b_ci_hi': 0.0148, 'e1_n': 2164, 'e1_ann': 0.584, 'e1_qtr': 0.555, 'e1_bnd': 0.535, 'e1_diff': -0.0491, 'e1_t': 0.15, 'e2_n': 2637, 'e2_ann': 0.741, 'e2_qtr': 0.751, 'e2_bnd': 0.724, 'e2_diff': -0.017, 'e2_t': 0.83, 'c0_bnd': 0.6416, 'c0_diff': -0.026, 'c0_t': 0.83, 'c50_ann': 0.664, 'c50_qtr': 0.6505, 'c50_bnd': 0.6367, 'c50_diff': -0.0273, 'c50_t': 0.73, 'drag50_ann': 3.94, 'drag50_qtr': 7.86, 'drag50_bnd': 5.69, 'bw': [(2, 10, 0.6346, 98, 0.221, -0.0327, 0.38), (3, 15, 0.6334, 44, 0.146, -0.0339, 0.31), (5, 25, 0.6412, 22, 0.114, -0.0261, 0.82), (8, 40, 0.6544, 9, 0.076, -0.0129, 0.93), (10, 50, 0.6488, 6, 0.063, -0.0185, 1.42)], 'w_drift_min': 0.357, 'w_drift_max': 0.85, 'w_drift_fin': 0.846, 'w_drift_dev': 0.1016, 'w_ann_min': 0.418, 'w_ann_max': 0.672, 'w_ann_dev': 0.0219, 'w_qtr_min': 0.481, 'w_qtr_max': 0.666, 'w_qtr_dev': 0.0133, 'w_bnd_min': 0.536, 'w_bnd_max': 0.652, 'w_bnd_dev': 0.0178, 'l_start': '2003-01-03', 'l_n': 5909, 'l_fp': '7788344e5927', 'l_drift': 0.812, 'l_ann': 0.8682, 'l_qtr': 0.8558, 'l_bnd': 0.8361, 'l_diff_ba': -0.0321, 'l_t_ba': 1.04, 'syn_pl_diff': 0.0414, 'syn_pl_sd': 0.0093, 'syn_pl_fire': 7, 'syn_nl_diff': 0.0044, 'syn_nl_sd': 0.0142, 'syn_nl_fire': 0, 'claim_pp': 0.5}

## Book A — 60/40 SPY/IEF, excess-of-cash, 5 bps one-way

In [2]:
hdr = f"{'schedule':<11}{'exSharpe':>10}{'CAGR':>9}{'vol':>8}{'maxDD':>9}"
hdr += f"{'traded/yr':>11}{'rebals':>8}"
print(hdr)
rows = [('drift', R['a_drift_s'], R['a_drift_c'], R['a_drift_v'], R['a_drift_dd'], 0.0, 0),
        ('annual', R['a_ann_s'], R['a_ann_c'], R['a_ann_v'], R['a_ann_dd'], R['a_ann_to'], R['a_ann_n']),
        ('quarterly', R['a_qtr_s'], R['a_qtr_c'], R['a_qtr_v'], R['a_qtr_dd'], R['a_qtr_to'], R['a_qtr_n']),
        ('bands', R['a_bnd_s'], R['a_bnd_c'], R['a_bnd_v'], R['a_bnd_dd'], R['a_bnd_to'], R['a_bnd_n'])]
for name, s, c, v, dd, to, n in rows:
    print(f'{name:<11}{s:>+10.4f}{c:>8.2f}%{v:>7.2f}%{dd:>8.2f}%{to:>11.3f}{n:>8d}')
print()
for tag, d, t in [('bands - annual', R['a_diff_ba'], R['a_t_ba']),
                  ('bands - quarterly', R['a_diff_bq'], R['a_t_bq']),
                  ('quarterly - annual', R['a_diff_qa'], R['a_t_qa'])]:
    print(f'{tag:<20} Sharpe diff {d:+.4f}   HAC t (return diff) {t:+.2f}')

schedule     exSharpe     CAGR     vol    maxDD  traded/yr  rebals
drift         +0.6788    7.23%  11.22%  -24.92%      0.000       0
annual        +0.6673    6.79%  10.71%  -30.01%      0.079      19
quarterly     +0.6570    6.84%  11.00%  -31.74%      0.157      76
bands         +0.6412    6.88%  11.40%  -31.86%      0.114      22

bands - annual       Sharpe diff -0.0261   HAC t (return diff) +0.82
bands - quarterly    Sharpe diff -0.0158   HAC t (return diff) +0.64
quarterly - annual   Sharpe diff -0.0103   HAC t (return diff) +0.53


> 💡 **In plain words:** the four ways of rebalancing land within three hundredths of a Sharpe point of each other over nineteen years. Whatever you were arguing about in the rebalancing chapter, it was not this.

The sign split is worth being explicit about. `bands - annual` has a **positive** HAC *t* (+0.82) on the daily return difference and a **negative** Sharpe difference (-0.0261): the band book earns marginally more per day and carries marginally more variance (11.40% vs 10.71% annualised). Both are noise. Reporting only the one with the convenient sign would be the classic way to manufacture a result here.

## Book B — 50/30/20 SPY/IEF/GLD

The 3-asset book is where the 5/25 rule's *relative* leg finally binds: the 20% gold sleeve gets a 2.5 pp band, not a 5 pp one. If the relative leg is what makes the rule special, it should show up here.

In [3]:
print(f"drift {R['b_drift_s']:+.4f}   annual {R['b_ann_s']:+.4f}   "
      f"quarterly {R['b_qtr_s']:+.4f}   bands {R['b_bnd_s']:+.4f}")
print(f"bands - annual: {R['b_diff_ba']:+.4f}  (HAC t {R['b_t_ba']:+.2f}, "
      f"CI [{R['b_ci_lo']:+.4f}, {R['b_ci_hi']:+.4f}])")
print(f"CAGR gap: {R['b_cagr_ba']:+.3f} pp/yr  vs the {R['claim_pp']:.1f} pp/yr claimed in the literature")

drift +0.7393   annual +0.7718   quarterly +0.7559   bands +0.7441
bands - annual: -0.0277  (HAC t +0.80, CI [-0.0685, +0.0148])
CAGR gap: +0.122 pp/yr  vs the 0.5 pp/yr claimed in the literature


## Paired block-bootstrap CI on the Sharpe difference

2,000 circular block resamples, 21-day blocks, the **same** block indices applied to both arms so the day-by-day pairing survives. Pairing is what makes these intervals tight: the common market factor cancels inside every resample.

In [4]:
print(f"60/40    bands - annual: {R['a_diff_ba']:+.4f}  "
      f"95% CI [{R['a_ci_lo']:+.4f}, {R['a_ci_hi']:+.4f}]  share<0 {R['a_ci_neg']:.1f}%")
print(f"50/30/20 bands - annual: {R['b_diff_ba']:+.4f}  "
      f"95% CI [{R['b_ci_lo']:+.4f}, {R['b_ci_hi']:+.4f}]")
width = R['a_ci_hi'] - R['a_ci_lo']
print(f'\ninterval width {width:.3f} Sharpe points -> a PRECISE null, not an underpowered test')

60/40    bands - annual: -0.0261  95% CI [-0.0674, +0.0142]  share<0 87.9%
50/30/20 bands - annual: -0.0277  95% CI [-0.0685, +0.0148]

interval width 0.082 Sharpe points -> a PRECISE null, not an underpowered test


## Era cut (split 2016-01-01), 60/40

The 2022 regime break flipped the stock/bond correlation, which is the single biggest driver of how far a 60/40 book drifts between rebalances. If the schedule choice ever mattered, it should matter differently on the two sides of that.

In [5]:
print(f"2007-2015 (n={R['e1_n']}): annual {R['e1_ann']:+.3f}  quarterly {R['e1_qtr']:+.3f}  "
      f"bands {R['e1_bnd']:+.3f}  |  bands-annual {R['e1_diff']:+.4f} (t={R['e1_t']:+.2f})")
print(f"2016-2026 (n={R['e2_n']}): annual {R['e2_ann']:+.3f}  quarterly {R['e2_qtr']:+.3f}  "
      f"bands {R['e2_bnd']:+.3f}  |  bands-annual {R['e2_diff']:+.4f} (t={R['e2_t']:+.2f})")
print('\nsame sign, same non-significance in both halves')

2007-2015 (n=2164): annual +0.584  quarterly +0.555  bands +0.535  |  bands-annual -0.0491 (t=+0.15)
2016-2026 (n=2637): annual +0.741  quarterly +0.751  bands +0.724  |  bands-annual -0.0170 (t=+0.83)

same sign, same non-significance in both halves


## Cost sweep — friction is not the binding constraint

In [6]:
print(f"  0 bps: bands {R['c0_bnd']:+.4f}   bands-annual {R['c0_diff']:+.4f} (t={R['c0_t']:+.2f})")
print(f" 50 bps: annual {R['c50_ann']:+.4f}  quarterly {R['c50_qtr']:+.4f}  "
      f"bands {R['c50_bnd']:+.4f}   bands-annual {R['c50_diff']:+.4f} (t={R['c50_t']:+.2f})")
print()
print('annual drag at 50 bps one-way (ten times realistic):')
print(f"  annual {R['drag50_ann']:.2f} bp/yr   quarterly {R['drag50_qtr']:.2f} bp/yr   "
      f"bands {R['drag50_bnd']:.2f} bp/yr")

  0 bps: bands +0.6416   bands-annual -0.0260 (t=+0.83)
 50 bps: annual +0.6640  quarterly +0.6505  bands +0.6367   bands-annual -0.0273 (t=+0.73)

annual drag at 50 bps one-way (ten times realistic):
  annual 3.94 bp/yr   quarterly 7.86 bp/yr   bands 5.69 bp/yr


> 💡 **In plain words:** the whole cost argument for tolerance bands is worth about four basis points a year, at ten times realistic trading costs. It is real and it is negligible — the folklore was formed when retail rebalancing meant loaded mutual funds, not penny-spread ETFs.

## The 5/25 widths are an ASSUMPTION — sweep them

5/25 is a memorable round number, not a calibrated parameter. If the result depended on it, that dependence would be the finding.

| Band | Excess Sharpe | Rebalances | Traded/yr | vs annual (*t*) |
|---|--:|--:|--:|--:|
| 2% / 10% | +0.6346 | 98 | 0.221 | -0.0327 (*t* = +0.38) |
| 3% / 15% | +0.6334 | 44 | 0.146 | -0.0339 (*t* = +0.31) |
| 5% / 25% | +0.6412 | 22 | 0.114 | -0.0261 (*t* = +0.82) |
| 8% / 40% | +0.6544 | 9 | 0.076 | -0.0129 (*t* = +0.93) |
| 10% / 50% | +0.6488 | 6 | 0.063 | -0.0185 (*t* = +1.42) |

A five-fold range of trigger widths moves the excess Sharpe by 0.02 with no monotone pattern beyond *trade less, keep a hair more*. 5/25 sits inside the flat region — neither optimal nor harmful.

## Weight discipline — the one axis where the schedules genuinely separate

In [7]:
print(f"{'schedule':<11}{'SPY min':>9}{'SPY max':>9}{'mean |dev|':>12}")
for name, lo, hi, dev in [('drift', R['w_drift_min'], R['w_drift_max'], R['w_drift_dev']),
                          ('annual', R['w_ann_min'], R['w_ann_max'], R['w_ann_dev']),
                          ('quarterly', R['w_qtr_min'], R['w_qtr_max'], R['w_qtr_dev']),
                          ('bands', R['w_bnd_min'], R['w_bnd_max'], R['w_bnd_dev'])]:
    print(f'{name:<11}{lo:>9.1%}{hi:>9.1%}{dev:>12.4f}')
print(f"\ndrift book final equity weight: {R['w_drift_fin']:.1%} "
      f"(it started at 60.0%)")
print(f"bands traded {1 - R['a_bnd_to']/R['a_qtr_to']:.0%} less notional than quarterly "
      f"for a TIGHTER envelope")
print(f"but ANNUAL is cheaper still ({R['a_ann_to']:.1%}/yr) at a much wider envelope,")
print(f"and QUARTERLY tracks target closer on an AVERAGE day "
      f"({R['w_qtr_dev']:.4f} vs {R['w_bnd_dev']:.4f}) — bands minimise the WORST excursion")

schedule     SPY min  SPY max  mean |dev|
drift          35.7%    85.0%      0.1016
annual         41.8%    67.2%      0.0219
quarterly      48.1%    66.6%      0.0133
bands          53.6%    65.2%      0.0178

drift book final equity weight: 84.6% (it started at 60.0%)
bands traded 27% less notional than quarterly for a TIGHTER envelope
but ANNUAL is cheaper still (7.9%/yr) at a much wider envelope,
and QUARTERLY tracks target closer on an AVERAGE day (0.0133 vs 0.0178) — bands minimise the WORST excursion


## Longer-window cross-check — SPY/IEF from 2003-01-03, gross of cash

BIL does not exist before 2007, so this window has **no tradable cash leg** and is run gross of cash; that buys four extra years of sample. The trap — and the first draft of this study walked into it — is comparing its Sharpe gap with the excess-of-cash headline. The cash leg cancels **exactly** in the daily *return* difference (the HAC *t* is bit-identical either way), but a Sharpe *ratio* is nonlinear, so the Sharpe *difference* moves. The cell below prints the matched comparator. (Returns fingerprint `7788344e5927`, n = 5,909.)

In [8]:
print('headline window 2007-2026, 60/40 — the SAME trades, two cash treatments:')
for tag, ex, gr in [('bands - annual', R['a_diff_ba'], R['a_diff_ba_gross']),
                    ('bands - quarterly', R['a_diff_bq'], R['a_diff_bq_gross']),
                    ('quarterly - annual', R['a_diff_qa'], R['a_diff_qa_gross'])]:
    print(f'  {tag:<20} excess {ex:+.4f}   gross {gr:+.4f}   shift {gr-ex:+.4f}')
print()
print(f"long window {R['l_start']}-2026 (gross of cash, n={R['l_n']:,}):")
print(f"  drift {R['l_drift']:+.4f}   annual {R['l_ann']:+.4f}   "
      f"quarterly {R['l_qtr']:+.4f}   bands {R['l_bnd']:+.4f}")
print(f"  bands - annual: {R['l_diff_ba']:+.4f}  (HAC t {R['l_t_ba']:+.2f})")
print(f"  vs the matched gross comparator {R['a_diff_ba_gross']:+.4f} on the headline window")
print('\n23 years, same answer — and, on a matched basis, a FRACTIONALLY SMALLER')
print('adverse gap than the headline window, not a larger one')

headline window 2007-2026, 60/40 — the SAME trades, two cash treatments:
  bands - annual       excess -0.0261   gross -0.0340   shift -0.0079
  bands - quarterly    excess -0.0158   gross -0.0204   shift -0.0046
  quarterly - annual   excess -0.0103   gross -0.0136   shift -0.0033

long window 2003-01-03-2026 (gross of cash, n=5,909):
  drift +0.8120   annual +0.8682   quarterly +0.8558   bands +0.8361
  bands - annual: -0.0321  (HAC t +1.04)
  vs the matched gross comparator -0.0340 on the headline window

23 years, same answer — and, on a matched basis, a FRACTIONALLY SMALLER
adverse gap than the headline window, not a larger one


## Live synthetic control — the harness reads the sign correctly

**Synthetic tape, not the real one.** Two sleeves whose idiosyncratic components follow an AR(1): at `signal_strength=1` dispersion mean-reverts with a 60-day half-life (acting on dispersion *should* pay — the planted world); at `signal_strength=0` dispersion is a random walk (nothing to harvest — the null). The panel is deliberately favourable to bands: equal drifts, equal vols, almost no common factor. If the band rule could not win *there*, the detector would be broken.

In [9]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from rebal_bands import data, strategy as st
for tag, ss in [('planted (reverting)', 1.0), ('null (random walk)', 0.0)]:
    diffs, ts = [], []
    for s in range(4):
        d = st.synthetic_detect(data.synthetic_panel(signal_strength=ss, seed=936+s)[0])
        diffs.append(d['diff_bands_annual']); ts.append(d['t_bands_annual'])
    diffs, ts = np.array(diffs), np.array(ts)
    print(f"{tag:22s}: bands-annual {diffs.mean():+.4f} (sd {diffs.std(ddof=1):.4f}), "
          f"HAC t >= 2 in {(ts >= 2).sum()}/4 seeds")

planted (reverting)   : bands-annual +0.0462 (sd 0.0043), HAC t >= 2 in 4/4 seeds


null (random walk)    : bands-annual +0.0098 (sd 0.0157), HAC t >= 2 in 0/4 seeds


Over the full 8-seed run in `docs/results.md`: planted **+0.0414** (sd 0.0093), fires **7/8**; null **+0.0044** (sd 0.0142), fires **0/8**. The detector recovers a planted effect and is silent on the null, so the real-tape flatness is a property of the asset tape: **stock/bond/gold relative performance does not mean-revert on the horizon a 5-point band operates over.** That is the economic reason the rule has nothing to harvest, and it is why no amount of band tuning rescues it.

## Verdict

- **Signal — None.** Across two books, two windows, two eras, six cost levels and five band widths, no schedule difference clears |*t*| = 2 in any specification. Headline: bands − annual **-0.0261** excess Sharpe, HAC *t* on the daily return difference **+0.82**, paired bootstrap CI **[-0.0674, +0.0142]** (share < 0 = 87.9%). 3-asset book -0.0277 (*t* = +0.80); 2003-2026 cross-check -0.0321 *gross-of-cash* (*t* = +1.04) against a -0.0340 gross-of-cash comparator on the headline window. Daryanani's ~0.5 pp/yr reproduces as **+0.09 pp/yr**. Intervals are narrow — a precise null. Survivorship: the sleeves are surviving mega-ETFs picked with hindsight, which can only flatter a result, and there is no result to flatter.
- **Tradability — Mirage.** Turnover is 7-17% of NAV/yr under every schedule; at a punitive 50 bps one-way the schedules separate by under 4 bp/yr. The only first-order unmodelled cost is **tax**, and it penalises the higher-turnover schedules — so modelling it would widen the gap *against* quarterly, not open one for bands.
- **What survives.** Weight discipline, on one statistic. 5/25 bands hold the equity sleeve inside 53.6%-65.2% for 27% less traded notional than quarterly, while the do-nothing book finishes at 84.6% equity. State the statistic honestly, though: annual trades *less* than bands (7.9% vs 11.4%) and quarterly tracks target closer *on average* (0.0133 vs 0.0178). Bands win the **worst-excursion** contest and nothing else. Choose the rule for the risk profile you want to keep, and do not budget a return for it.